# Management of a short strangle

Chapter 6 of *The Unlucky Investor's Guide to Options Trading* compares holding a trade to expiry with closing it early. This notebook runs the 16 delta strangle at 45 DTE (`options/premium/management.py`) four ways: held to expiry, closed at 21 DTE, closed at 50 % of the credit, and closed at whichever comes first. A profit target of 10 000 % never triggers.

Set `SBT2_SERVER_TOKEN`, and `SBT2_SERVER_URL` if the server is not on this machine. The server needs the Deribit data of the period.

In [ ]:
import os

import pandas as pd

from sbt2.lab import Lab

url = os.environ.get("SBT2_SERVER_URL", "ws://127.0.0.1:8765")
lab = await Lab.connect(url, os.environ["SBT2_SERVER_TOKEN"])

In [ ]:
spec = {
    "instruments": ["BTC-OPTIONS.DERIBIT"],
    "period": ["2024-01-01", "2025-01-01"],
    "venue": "deribit_option",
    "capital": "1 BTC",
    "split": {"validation": 0.2, "test": 0.2},
    "part": "train",
    "chain": {"min_dte": 0, "max_dte": 60, "strike_band": "0.3"},
}

runs = await lab.run(
    {**spec, "params": {"close_dte": [0, 21], "profit_target": [10_000, 50]}},
    "options.premium.management:ShortStrangle",
)
runs.table()

The statistics over the trades of each run: the probability of profit (`POP`), the tail loss and the buying power reduction.

In [ ]:
rows = {}
for run in runs:
    key = (run.params["close_dte"], run.params["profit_target"])
    rows[key] = await run.trade_statistics()
pd.DataFrame(rows).T.rename_axis(["close_dte", "profit_target"])

In [ ]:
await runs[-1].tearsheet()

In [ ]:
await lab.close()